In [ ]:
!python -m pip install -q ir_datasets "pyserini==2.3.0" pytrec_eval tqdm pandas

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-21-jdk

In [ ]:
!python -m pip uninstall -y Pillow
!python -m pip install --no-cache-dir "Pillow==12.3.0" 

# Then restart the kernel . 

In [1]:
import os
import json
import time
import shutil
import subprocess
import gc

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import ir_datasets
import pytrec_eval

from tqdm.auto import tqdm
from pyserini.search.lucene import LuceneSearcher

In [38]:
# Configuration

BASE_DIR = Path("rankgpt_project")

DATA_ROOT = BASE_DIR / "data"
INDEX_ROOT = BASE_DIR / "indexes"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
INDEX_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_CONFIGS = [

    {
        "name": "nfcorpus",

        # Contains the full 3,633-document corpus and the test queries.
        "corpus_dataset": "beir/nfcorpus/test",
        "query_dataset": "beir/nfcorpus/test",

        # Qrels we want to use for evaluation.
        "qrels_datasets": [
            "beir/nfcorpus/test"
        ],
    },

    {
        "name": "touche",

        "corpus_dataset": "beir/webis-touche2020",
        "query_dataset": "beir/webis-touche2020",

        "qrels_datasets": [
            "beir/webis-touche2020"
        ],
    },

    {
        "name": "trec-covid",

        "corpus_dataset": "beir/trec-covid",
        "query_dataset": "beir/trec-covid",

        "qrels_datasets": [
            "beir/trec-covid"
        ],
    },

    {
        "name": "dbpedia",

        "corpus_dataset": "beir/dbpedia-entity/test",
        "query_dataset": "beir/dbpedia-entity/test",

        "qrels_datasets": [
            "beir/dbpedia-entity/test"
        ],
    },

    {
        "name": "scifact",

        "corpus_dataset": "beir/scifact/test",
        "query_dataset": "beir/scifact/test",

        "qrels_datasets": [
            "beir/scifact/test"
        ],
    },
]

# Retreival Configuration

TOP_K = 100

# I am not changing this using default bm25 parameters.
BM25_K1 = None
BM25_B = None

INDEX_THREADS = 4

In [3]:
def get_dataset_paths(dataset_name):
    # It returns all dataset paths

    data_dir = DATA_ROOT / dataset_name
    index_dir = INDEX_ROOT / f"{dataset_name}_bm25"

    corpus_jsonl = data_dir / "corpus.jsonl"
    queries_jsonl = data_dir / "queries.jsonl"
    qrels_tsv = data_dir / "qrels.tsv"

    corpus_collection_dir = data_dir / "corpus_collection"

    bm25_top100_jsonl = data_dir / "bm25_top100.jsonl"
    reranking_data_jsonl = data_dir / "reranking_data.jsonl"

    bm25_metrics_csv = data_dir / "bm25_metrics.csv"

    return {
        "data_dir": data_dir,
        "index_dir": index_dir,
        "corpus_jsonl": corpus_jsonl,
        "queries_jsonl": queries_jsonl,
        "qrels_tsv": qrels_tsv,
        "corpus_collection_dir": corpus_collection_dir,
        "bm25_top100_jsonl": bm25_top100_jsonl,
        "reranking_data_jsonl": reranking_data_jsonl,
        "bm25_metrics_csv": bm25_metrics_csv,
    }

In [4]:
def export_corpus(dataset, output_file):

    doc_store = {}

    total_docs = dataset.docs_count()

    print(f"Exporting {total_docs:,} documents...")

    with open(output_file, "w", encoding="utf-8") as f:

        for doc in tqdm(dataset.docs_iter(),total=total_docs,desc="Writing corpus"):

            doc_id = str(doc.doc_id)

            title = getattr(doc, "title", "") or ""
            text = getattr(doc, "text", "") or ""

           
            doc_store[doc_id] = {
                "title": title,
                "text": text,
            }

            # Text used by BM25.
            contents = f"{title} {text}".strip()

            record = {
                "id": doc_id,
                "contents": contents,
            }

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

    print(f"Corpus saved to: {output_file}")
    print(f"Documents loaded: {len(doc_store):,}")

    return doc_store

In [6]:
def load_queries(dataset):

    queries = {}

    total_queries = dataset.queries_count()

    for q in tqdm(dataset.queries_iter(),total=total_queries,desc="Loading queries"):

        query_id = str(q.query_id)

        queries[query_id] = {
            "query_id": query_id,
            "query_text": q.text,
        }

    print(f"Queries loaded: {len(queries):,}")

    return queries


def save_queries(queries, output_file, split_name=None):

    with open(output_file, "w", encoding="utf-8") as f:

        for query_id, query in queries.items():

            record = {
                "query_id": query_id,
                "query_text": query["query_text"],
            }

            if split_name is not None:
                record["split"] = split_name

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

    print(f"Queries saved to: {output_file}")

In [7]:
def load_qrels(qrels_dataset_ids):

    qrels = defaultdict(dict)

    for dataset_id in qrels_dataset_ids:

        print(f"\nLoading qrels from: {dataset_id}")

        dataset = ir_datasets.load(dataset_id)

        count = 0

        for qrel in tqdm(
            dataset.qrels_iter(),
            desc=f"Qrels: {dataset_id}"
        ):

            query_id = str(qrel.query_id)
            doc_id = str(qrel.doc_id)
            relevance = int(qrel.relevance)

            qrels[query_id][doc_id] = relevance

            count += 1

        print(f"Loaded {count:,} qrel entries.")

    total_pairs = sum(len(doc_dict) for doc_dict in qrels.values())

    print(f"\nQueries with qrels: {len(qrels):,}")
    print(f"Total qrel pairs:   {total_pairs:,}")

    return qrels

In [8]:
def save_qrels(qrels, output_file):

    with open(output_file, "w", encoding="utf-8") as f:

        for query_id in sorted(qrels.keys()):

            for doc_id, relevance in sorted(
                qrels[query_id].items()
            ):

                f.write(
                    f"{query_id}\t0\t"
                    f"{doc_id}\t{relevance}\n"
                )

    print(f"Qrels saved to: {output_file}")

In [9]:
def build_bm25_index(corpus_collection_dir,corpus_jsonl,index_dir):

    corpus_collection_dir.mkdir(parents=True,exist_ok=True)

    corpus_for_index = corpus_collection_dir / "corpus.jsonl"

    # Pyserini should see only corpus.jsonl.
    if not corpus_for_index.exists():

        shutil.copy2(corpus_jsonl,corpus_for_index)

    # Don't rebuild an existing index.
    if index_dir.exists() and any(index_dir.iterdir()):

        print(f"Index already exists:")
        print(index_dir)
        print("Skipping index construction.")

        return

    print("\nBuilding Lucene BM25 index...")

    start_time = time.time()

    cmd = [
        "python",
        "-m",
        "pyserini.index.lucene",

        "--collection",
        "JsonCollection",

        "--input",
        str(corpus_collection_dir),

        "--index",
        str(index_dir),

        "--generator",
        "DefaultLuceneDocumentGenerator",

        "--threads",
        str(INDEX_THREADS),

        "--storePositions",
        "--storeDocvectors",
        "--storeRaw",
    ]

    subprocess.run(cmd,check=True)

    elapsed = time.time() - start_time

    print(
        f"Index built successfully "
        f"in {elapsed:.2f} seconds."
    )

In [10]:
def retrieve_bm25(queries,index_dir,top_k=100):
    # Retrieve Top-K docs for each query

    searcher = LuceneSearcher(str(index_dir))

    if BM25_K1 is not None and BM25_B is not None:
        searcher.set_bm25(BM25_K1,BM25_B)

    print(f"Indexed documents: {searcher.num_docs:,}")

    bm25_results = {}

    start_time = time.time()

    for query_id, query_info in tqdm(queries.items(),total=len(queries),desc="BM25 retrieval"):

        hits = searcher.search(query_info["query_text"],k=top_k)

        results = []

        for rank, hit in enumerate(hits,start=1):

            results.append({
                "rank": rank,
                "doc_id": str(hit.docid),
                "bm25_score": float(hit.score),
            })

        # We may also get less than 100 results.
        bm25_results[query_id] = results

    elapsed = time.time() - start_time

    total_candidates = sum(
        len(results)
        for results in bm25_results.values()
    )

    print(f"\nRetrieved BM25 results for {len(bm25_results):,} queries.")

    print(f"Total candidate pairs: {total_candidates:,}")

    print(f"Elapsed time: {elapsed:.2f} seconds")

    return bm25_results

In [12]:
def save_bm25_results(queries,bm25_results,output_file):

    with open(output_file, "w", encoding="utf-8") as f:

        for query_id, results in bm25_results.items():

            record = {
                "query_id": query_id,
                "query_text": queries[query_id]["query_text"],
                "results": results,
            }

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

    print(f"BM25 results saved to: {output_file}")

In [22]:
def build_reranking_dataset(queries,qrels,bm25_results,doc_store,output_file):
    
    print("\nBuilding master reranking dataset...")

    start_time = time.time()

    # Count positive gold documents whose text is unavailable
    missing_gold_docs = 0

    with open(output_file, "w", encoding="utf-8") as f:

        for query_id, query_info in tqdm(queries.items(),total=len(queries),desc="Building reranking data"):

            # Gold relevance judgments for this query
            query_qrels = qrels.get(query_id,{})

            retrieved_docs = []

            for result in bm25_results.get(query_id,[]):

                doc_id = str(result["doc_id"])
                
                if doc_id not in doc_store:

                    raise KeyError(
                        f"Retrieved document {doc_id} not found in corpus.")

                doc_info = doc_store[doc_id]

                relevance = int(query_qrels.get(doc_id,0))

                retrieved_docs.append({

                    "rank": int(result["rank"]),

                    "doc_id": doc_id,

                    "bm25_score": float(result["bm25_score"]),

                    "title": doc_info["title"],

                    "text": doc_info["text"],

                    "relevance": relevance
                })



            relevant_docs = []

            for doc_id, relevance in query_qrels.items():

                relevance = int(relevance)

                # Only positive relevance judgments are
                # considered gold relevant documents.
                if relevance <= 0:
                    continue

                doc_id = str(doc_id)

                # --------------------------------------------------------
                # Gold document is available in our corpus
                # --------------------------------------------------------

                if doc_id in doc_store:

                    doc_info = doc_store[doc_id]

                    relevant_docs.append({

                        "doc_id":
                            doc_id,

                        "title":
                            doc_info["title"],

                        "text":
                            doc_info["text"],

                        "relevance":
                            relevance,

                        "corpus_available":
                            True
                    })

                # --------------------------------------------------------
                # Gold document referenced by qrels but absent
                # from the corpus
                # --------------------------------------------------------

                else:

                    missing_gold_docs += 1

                    relevant_docs.append({

                        "doc_id":
                            doc_id,

                        "title":
                            None,

                        "text":
                            None,

                        "relevance":
                            relevance,

                        "corpus_available":
                            False
                    })



            # Positive gold document IDs
            gold_doc_ids = {
                str(doc_id)
                for doc_id, relevance
                in query_qrels.items()
                if int(relevance) > 0
            }

            # Documents returned by BM25
            retrieved_doc_ids = {doc["doc_id"] for doc in retrieved_docs}

            # Gold documents that BM25 actually retrieved
            retrieved_gold = (gold_doc_ids &retrieved_doc_ids)

            num_gold = len(gold_doc_ids)

            if num_gold > 0:

                recall_at_100 = (len(retrieved_gold)/ num_gold)

            else:

                recall_at_100 = None


            gold_bm25_ranks = {}

            for doc in retrieved_docs:

                doc_id = doc["doc_id"]

                if doc_id in gold_doc_ids:

                    gold_bm25_ranks[doc_id] = doc["rank"]


            # Final query record

            record = {

                "query_id": query_id,

                "query_text": query_info["query_text"],

                "retrieved_docs":
                    retrieved_docs,

                "relevant_docs":
                    relevant_docs,

                "stage1_stats": {

                    "num_retrieved":
                        len(retrieved_docs),

                    "num_gold_docs":
                        num_gold,

                    "num_gold_retrieved":
                        len(retrieved_gold),

                    "recall_at_100":
                        recall_at_100,

                    "gold_bm25_ranks":
                        gold_bm25_ranks
                }
            }


            # Write one complete query record
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )


    # Final SUmmary 

    elapsed = time.time() - start_time

    print(f"\nMaster dataset saved to: \n{output_file}")

    print(f"Build time: {elapsed:.2f} seconds")

    print(f"Missing positive gold documents: {missing_gold_docs}")

In [14]:
def evaluate_bm25(qrels,bm25_results):

    # Convert qrels
    qrels_eval = {
        str(qid): {
            str(docid): int(relevance)
            for docid, relevance
            in doc_dict.items()
        }
        for qid, doc_dict in qrels.items()
    }

    # Convert bm25 results
    bm25_run = {
        str(qid): {
            str(result["doc_id"]):
                float(result["bm25_score"])

            for result in query_results
        }

        for qid, query_results
        in bm25_results.items()
    }

    evaluator = pytrec_eval.RelevanceEvaluator(
        qrels_eval,
        {
            "ndcg_cut.1",
            "ndcg_cut.10",
            "recall.100",
            "map",
        }
    )

    eval_results = evaluator.evaluate(bm25_run)

    num_queries = len(eval_results)


    mean_ndcg1 = sum( result["ndcg_cut_1"] for result in eval_results.values() ) / num_queries

    mean_ndcg10 = sum( result["ndcg_cut_10"] for result in eval_results.values() ) / num_queries

    mean_recall100 = sum(result["recall_100"] for result in eval_results.values()) / num_queries

    mean_map = sum(result["map"] for result in eval_results.values()) / num_queries

    mrr_scores = []

    for qid, hits in bm25_run.items():

        relevant_docs = {
            str(docid)

            for docid, relevance
            in qrels_eval.get(qid, {}).items()

            if relevance > 0
        }

        ranked_hits = sorted(
            hits.items(),
            key=lambda x: x[1],
            reverse=True
        )[:10]

        reciprocal_rank = 0.0

        for rank, (docid, score) in enumerate(ranked_hits,start=1):

            if docid in relevant_docs:
                reciprocal_rank = 1.0 / rank
                break

        mrr_scores.append(reciprocal_rank)

    mean_mrr10 = (sum(mrr_scores) / len(mrr_scores) if mrr_scores else 0.0)

    # Final aggregrate metric 
    
    metrics = {
        "Queries": num_queries,
        "nDCG@1": mean_ndcg1,
        "nDCG@10": mean_ndcg10,
        "Recall@100": mean_recall100,
        "MAP": mean_map,
        "MRR@10": mean_mrr10,
    }

    return metrics

In [15]:
def print_dataset_summary(dataset_name,doc_store,queries,qrels,bm25_results):

    candidate_counts = [len(results) for results in bm25_results.values()]

    print("\n")
    print("=" * 70)
    print(f"{dataset_name.upper()} DATASET SUMMARY")
    print("=" * 70)

    print(
        f"Documents              : "
        f"{len(doc_store):,}"
    )

    print(
        f"Queries                : "
        f"{len(queries):,}"
    )

    print(
        f"Queries with qrels     : "
        f"{len(qrels):,}"
    )

    print(
        f"Total qrel pairs       : "
        f"{sum(len(x) for x in qrels.values()):,}"
    )

    print(
        f"Min BM25 candidates    : "
        f"{min(candidate_counts):,}"
    )

    print(
        f"Max BM25 candidates    : "
        f"{max(candidate_counts):,}"
    )

    print(
        f"Mean BM25 candidates   : "
        f"{np.mean(candidate_counts):.2f}"
    )

    print(
        f"Queries with < {TOP_K} results: "
        f"{sum(c < TOP_K for c in candidate_counts)}"
    )

    print("=" * 70)

In [16]:
def process_dataset(config):

    dataset_name = config["name"]

    print("\n\n")
    print("#" * 80)
    print(f"# PROCESSING DATASET: {dataset_name.upper()}")
    print("#" * 80)

    paths = get_dataset_paths(dataset_name)

    # Create directories
    paths["data_dir"].mkdir(parents=True,exist_ok=True)

    # Load Dataset
    corpus_dataset = ir_datasets.load(config["corpus_dataset"])
    query_dataset = ir_datasets.load(config["query_dataset"])

    # Prepare Corpus
    print("\n[1/7] Preparing corpus")

    doc_store = export_corpus(corpus_dataset,paths["corpus_jsonl"])

    # Prepare Queries
    print("\n[2/7] Preparing queries")
    queries = load_queries(query_dataset)
    save_queries(queries,paths["queries_jsonl"])

    # Prepare Qurels
    print("\n[3/7] Preparing qrels")
    qrels = load_qrels(config["qrels_datasets"])
    save_qrels(qrels,paths["qrels_tsv"])


    # Build Index
    print("\n[4/7] Building BM25 index")

    build_bm25_index(
        paths["corpus_collection_dir"],
        paths["corpus_jsonl"],
        paths["index_dir"],
    )


    # BM25 Retrieval
    print("\n[5/7] Running BM25")

    bm25_results = retrieve_bm25(
        queries,
        paths["index_dir"],
        top_k=TOP_K
    )

    save_bm25_results(
        queries,
        bm25_results,
        paths["bm25_top100_jsonl"]
    )

    print_dataset_summary(
        dataset_name,
        doc_store,
        queries,
        qrels,
        bm25_results,
    )

    # Build master reranking dataset
    print("\n[6/7] Building master reranking dataset")

    build_reranking_dataset(
        queries,
        qrels,
        bm25_results,
        doc_store,
        paths["reranking_data_jsonl"],
    )

    # Evaluate BM25 
    print("\n[7/7] Evaluating BM25")

    metrics = evaluate_bm25(qrels,bm25_results)

    final_table = pd.DataFrame([{
        "Dataset": dataset_name,
        **metrics
    }])

    display(
        final_table.style.format({
            "nDCG@1": "{:.4f}",
            "nDCG@10": "{:.4f}",
            "Recall@100": "{:.4f}",
            "MAP": "{:.4f}",
            "MRR@10": "{:.4f}",
        })
    )

    # Save metrics
    final_table.to_csv(paths["bm25_metrics_csv"],index=False)

    print(
        f"BM25 metrics saved to:"
        f"\n{paths['bm25_metrics_csv']}"
    )

    return {
        "paths": paths,
        "doc_store": doc_store,
        "queries": queries,
        "qrels": qrels,
        "bm25_results": bm25_results,
        "metrics": metrics,
    }

In [17]:
nfcorpus_config = next(cfg for cfg in DATASET_CONFIGS if cfg["name"] == "nfcorpus")

nfcorpus_output = process_dataset(nfcorpus_config)

[INFO] [starting] building docstore





################################################################################
# PROCESSING DATASET: NFCORPUS
################################################################################

[1/7] Preparing corpus
Exporting 3,633 documents...


[INFO] [starting] opening zip file                                              
[INFO] [starting] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip
docs_iter:   0%|                                      | 0/3633 [00:00<?, ?doc/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip: 0.0%| 0.00/2.45M [00:00<?, ?B/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip: 1.3%| 32.8k/2.45M [00:00<00:07, 307kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip: 3.3%| 81.9k/2.45M [00:00<00:06, 372kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip: 7.4%| 180k/2.45M [00:00<00:04, 541kB/s] 
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip: 16.1%| 393k/2.45M [00:00<00:02, 882kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip: 32.8%| 803k/2.45M [00:00<00:01, 1.44MB/s]
               

Writing corpus:   0%|          | 0/3633 [00:00<?, ?it/s]

[INFO] Opening /root/.ir_datasets/beir/nfcorpus/docs.pklz4/bin with direct file access


...


Retrieved BM25 results for 323 queries.
Total candidate pairs: 22,028
Elapsed time: 2.70 seconds
BM25 results saved to: rankgpt_project/data/nfcorpus/bm25_top100.jsonl


NFCORPUS DATASET SUMMARY
Documents              : 3,633
Queries                : 323
Queries with qrels     : 323
Total qrel pairs       : 12,334
Min BM25 candidates    : 0
Max BM25 candidates    : 100
Mean BM25 candidates   : 68.20
Queries with < 100 results: 123

[6/7] Building master reranking dataset

Building master reranking dataset...


Building reranking data:   0%|          | 0/323 [00:00<?, ?it/s]


Master dataset saved to: rankgpt_project/data/nfcorpus/reranking_data.jsonl
Build time: 0.88 seconds

[7/7] Evaluating BM25


,Dataset,Queries,nDCG@1,nDCG@10,Recall@100,MAP,MRR@10
0,nfcorpus,323,0.4257,0.3218,0.2457,0.1450,0.5238


BM25 metrics saved to:
rankgpt_project/data/nfcorpus/bm25_metrics.csv


In [19]:
covid_config = next(cfg for cfg in DATASET_CONFIGS if cfg["name"] == "trec-covid")

covid_output = process_dataset(nfcorpus_config)

[INFO] [starting] building docstore





################################################################################
# PROCESSING DATASET: TREC-COVID
################################################################################

[1/7] Preparing corpus
Exporting 171,332 documents...


[INFO] [starting] opening zip file                                              
[INFO] If you have a local copy of https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/trec-covid.zip, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/ce62140cb23feb9becf6270d0d1fe6d1
[INFO] [starting] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/trec-covid.zip
docs_iter:   0%|                                    | 0/171332 [00:01<?, ?doc/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/trec-covid.zip: 0.0%| 0.00/73.9M [00:00<?, ?B/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/trec-covid.zip: 0.0%| 32.8k/73.9M [00:00<03:48, 323kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/trec-covid.zip: 0.1%| 81.9k/73.9M [00:00<03:09, 390kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/trec-covid.zip: 0.3%| 197k/73.9M [00:00<01:59, 617kB/s] 
https://public.

Writing corpus:   0%|          | 0/171332 [00:00<?, ?it/s]

[INFO] Opening /root/.ir_datasets/beir/trec-covid/docs.pklz4/bin with direct file access


...


Retrieved BM25 results for 50 queries.
Total candidate pairs: 5,000
Elapsed time: 0.79 seconds
BM25 results saved to: rankgpt_project/data/trec-covid/bm25_top100.jsonl


TREC-COVID DATASET SUMMARY
Documents              : 171,332
Queries                : 50
Queries with qrels     : 50
Total qrel pairs       : 66,336
Min BM25 candidates    : 100
Max BM25 candidates    : 100
Mean BM25 candidates   : 100.00
Queries with < 100 results: 0

[6/7] Building master reranking dataset

Building master reranking dataset...


Building reranking data:   0%|          | 0/50 [00:00<?, ?it/s]


Master dataset saved to: rankgpt_project/data/trec-covid/reranking_data.jsonl
Build time: 0.71 seconds

[7/7] Evaluating BM25


,Dataset,Queries,nDCG@1,nDCG@10,Recall@100,MAP,MRR@10
0,trec-covid,50,0.6800,0.5947,0.1091,0.0734,0.8529


BM25 metrics saved to:
rankgpt_project/data/trec-covid/bm25_metrics.csv


In [32]:
dbpedia_config = next(cfg for cfg in DATASET_CONFIGS if cfg["name"] == "dbpedia")

dbpedia_output = process_dataset(dbpedia_config)

[INFO] [starting] building docstore





################################################################################
# PROCESSING DATASET: DBPEDIA
################################################################################

[1/7] Preparing corpus
Exporting 4,635,922 documents...


[INFO] [starting] opening zip file                                              
[INFO] If you have a local copy of https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/dbpedia-entity.zip, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/c2a39eb420a3164af735795df012ac2c
[INFO] [starting] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/dbpedia-entity.zip
docs_iter:   0%|                                   | 0/4635922 [00:01<?, ?doc/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/dbpedia-entity.zip: 0.0%| 0.00/639M [00:00<?, ?B/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/dbpedia-entity.zip: 0.0%| 32.8k/639M [00:00<34:02, 313kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/dbpedia-entity.zip: 0.0%| 81.9k/639M [00:00<28:08, 379kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/dbpedia-entity.zip: 0.0%| 197k/639M [00:00<17:44, 600kB

Writing corpus:   0%|          | 0/4635922 [00:00<?, ?it/s]

[INFO] Opening /root/.ir_datasets/beir/dbpedia-entity/docs.pklz4/bin with direct file access


...


Retrieved BM25 results for 400 queries.
Total candidate pairs: 39,905
Elapsed time: 6.21 seconds
BM25 results saved to: rankgpt_project/data/dbpedia/bm25_top100.jsonl


DBPEDIA DATASET SUMMARY
Documents              : 4,635,922
Queries                : 400
Queries with qrels     : 400
Total qrel pairs       : 43,515
Min BM25 candidates    : 5
Max BM25 candidates    : 100
Mean BM25 candidates   : 99.76
Queries with < 100 results: 1

[6/7] Building master reranking dataset

Building master reranking dataset...


Building reranking data:   0%|          | 0/400 [00:00<?, ?it/s]


Master dataset saved to: 
rankgpt_project/data/dbpedia/reranking_data.jsonl
Build time: 0.88 seconds
Missing positive gold documents: 0

[7/7] Evaluating BM25


,Dataset,Queries,nDCG@1,nDCG@10,Recall@100,MAP,MRR@10
0,dbpedia,400,0.3638,0.3180,0.4682,0.2219,0.5826


BM25 metrics saved to:
rankgpt_project/data/dbpedia/bm25_metrics.csv


In [40]:
scifact_config = next(cfg for cfg in DATASET_CONFIGS if cfg["name"] == "scifact")

scifact_output = process_dataset(scifact_config)

[INFO] [starting] building docstore





################################################################################
# PROCESSING DATASET: SCIFACT
################################################################################

[1/7] Preparing corpus
Exporting 5,183 documents...


[INFO] [starting] opening zip file                                              
[INFO] [starting] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip
docs_iter:   0%|                                      | 0/5183 [00:01<?, ?doc/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip: 0.0%| 0.00/2.82M [00:00<?, ?B/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip: 1.2%| 32.8k/2.82M [00:00<00:08, 315kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip: 2.9%| 81.9k/2.82M [00:00<00:07, 379kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip: 7.0%| 197k/2.82M [00:00<00:04, 599kB/s] 
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip: 14.0%| 393k/2.82M [00:00<00:02, 893kB/s]
https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip: 28.5%| 803k/2.82M [00:00<00:01, 1.45MB/s]
https://public.ukp.inf

Writing corpus:   0%|          | 0/5183 [00:00<?, ?it/s]

[INFO] Opening /root/.ir_datasets/beir/scifact/docs.pklz4/bin with direct file access


...


Retrieved BM25 results for 300 queries.
Total candidate pairs: 29,966
Elapsed time: 2.46 seconds
BM25 results saved to: rankgpt_project/data/scifact/bm25_top100.jsonl


SCIFACT DATASET SUMMARY
Documents              : 5,183
Queries                : 300
Queries with qrels     : 300
Total qrel pairs       : 339
Min BM25 candidates    : 66
Max BM25 candidates    : 100
Mean BM25 candidates   : 99.89
Queries with < 100 results: 1

[6/7] Building master reranking dataset

Building master reranking dataset...


Building reranking data:   0%|          | 0/300 [00:00<?, ?it/s]


Master dataset saved to: 
rankgpt_project/data/scifact/reranking_data.jsonl
Build time: 0.86 seconds
Missing positive gold documents: 0

[7/7] Evaluating BM25


,Dataset,Queries,nDCG@1,nDCG@10,Recall@100,MAP,MRR@10
0,scifact,300,0.5533,0.6789,0.9253,0.6398,0.6457


BM25 metrics saved to:
rankgpt_project/data/scifact/bm25_metrics.csv
